**Import libraries**

In [ ]:
import os
os.chdir("/exp/sbnd/app/users/svidales/AI_nuvT")

In [2]:
from imports import *

2025-05-05 14:42:02.501458: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-05 14:42:03.262804: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-05 14:42:04.333759: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-05 14:42:09.500453: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## 1. Preprocessing

The data corresponds to a MC simulation of the SBND experiment used in the paper "Scintillation Light in SBND: Simulation, Reconstruction, and Expected Performance of the Photon Detection System" in https://arxiv.org/abs/2406.07514. It is a simulation of BNB + Cosmics and their subsequent interaction in SBND, as well as the simulation of the detector's response to the particles resulting from the interaction of the neutrinos.

### Load variables

**archivos en espacio personal - a partir de ahora correrlos en data**

In [3]:
file_path = '/exp/sbnd/data/users/svidales/opana_tree_combined_v2304_complete.root'
file = uproot.open(file_path)
optree = file['opanatree']['OpAnaTree'] # Tree con número de fotoelectrones
print("Keys in optree:", optree.keys())

Keys in optree: ['eventID', 'runID', 'subrunID', 'nuvX', 'nuvY', 'nuvZ', 'nuvT', 'nuvE', 'stepX', 'stepY', 'stepZ', 'stepT', 'dE', 'energydep', 'energydepX', 'energydepY', 'energydepZ', 'E', 'StartPx', 'StartPy', 'StartPz', 'EndPx', 'EndPy', 'EndPz', 'process', 'trackID', 'motherID', 'PDGcode', 'InTimeCosmics', 'InTimeCosmicsTime', 'dEtpc', 'dEpromx', 'dEpromy', 'dEpromz', 'dEspreadx', 'dEspready', 'dEspreadz', 'dElowedges', 'dEmaxedges', 'nopflash', 'flash_id', 'flash_time', 'flash_total_pe', 'flash_pe_v', 'flash_tpc', 'flash_y', 'flash_yerr', 'flash_z', 'flash_zerr', 'flash_x', 'flash_xerr', 'flash_ophit_time', 'flash_ophit_risetime', 'flash_ophit_starttime', 'flash_ophit_amp', 'flash_ophit_area', 'flash_ophit_width', 'flash_ophit_pe', 'flash_ophit_ch']


In [4]:
# Input variables
# f_ophit_PE: number of photoelectrons (PEs) per OpHit
# f_ophit_ch: number of channel that collect the OpHit
# f_ophit_t: OpHit time

# Labels
# nuvT: neutrino interaction time
# dEprom{x,y,z}: energy barycenter coordinates {x,y,z}

# Auxiliary variables
# nuvZ: z-coordinate of the neutrino interaction
# dEtpc: allows filtering by energy deposited in the TPC

# These are awkward arrays (i.e., irregular structures) with a 3-level hierarchy (events -> flashes -> hits)

f_ophit_PE, f_ophit_ch, f_ophit_t, nuvT, dEpromx, dEpromy, dEpromz, dEtpc, nuvZ = (
    optree[key].array() for key in 
    ['flash_ophit_pe', 'flash_ophit_ch', 'flash_ophit_time', 'nuvT', 'dEpromx', 'dEpromy', 'dEpromz', 'dEtpc', "nuvZ"]
)

In [5]:
print(len(f_ophit_PE))

243599


**Eliminate events with more than one neutrino & events with no flashes**

In [6]:
# Filter events where nuvT has exactly one element
mask = ak.num(nuvT) == 1  

# Apply the mask to all variables
f_ophit_PE_1, f_ophit_ch_1, f_ophit_t_1 = f_ophit_PE[mask], f_ophit_ch[mask], f_ophit_t[mask]
nuvT_1, dEpromx_1, dEpromy_1, dEpromz_1, dEtpc_1, nuvZ_1 = nuvT[mask], dEpromx[mask], dEpromy[mask], dEpromz[mask], dEtpc[mask], nuvZ[mask]

In [8]:
print(len(f_ophit_PE_1))

102618


In [7]:
del f_ophit_PE, f_ophit_ch, f_ophit_t, nuvT, dEpromx, dEpromy, dEpromz, dEtpc, nuvZ

In [9]:
# Filter events with at least one flash
mask = ak.num(f_ophit_PE_1, axis=1) > 0  

# Apply the mask to all variables
f_ophit_PE_2, f_ophit_ch_2, f_ophit_t_2 = f_ophit_PE_1[mask], f_ophit_ch_1[mask], f_ophit_t_1[mask]
nuvT_2, dEpromx_2, dEpromy_2, dEpromz_2, dEtpc_2, nuvZ_2 = nuvT_1[mask], dEpromx_1[mask], dEpromy_1[mask], dEpromz_1[mask], dEtpc_1[mask], nuvZ_1[mask]


In [10]:
print(len(f_ophit_PE_2))

101189


In [11]:
del f_ophit_PE_1, f_ophit_ch_1, f_ophit_t_1, nuvT_1, dEpromx_1, dEpromy_1, dEpromz_1, dEtpc_1, nuvZ_1

### Corrección PTM delay

**Correction PMT delay 135 ns due to the difference between the photon arrival times (at the photocathode)
and the digitised signal (at the anode)**

**es posible que luego añada las coordenadas en la siguiente celda y lo guarde**

In [12]:
PDSMap = file['opanatree']['PDSMapTree']
ID = PDSMap['OpDetID'].array()
Type = PDSMap['OpDetType'].array()
channel_dict = {id_val: (int(type_val)) for id_val, type_val in zip(ID[0],Type[0])}
print(channel_dict)

{np.int32(0): 3, np.int32(1): 3, np.int32(2): 3, np.int32(3): 3, np.int32(4): 3, np.int32(5): 3, np.int32(6): 0, np.int32(7): 0, np.int32(8): 0, np.int32(9): 0, np.int32(10): 0, np.int32(11): 0, np.int32(12): 0, np.int32(13): 0, np.int32(14): 0, np.int32(15): 0, np.int32(16): 0, np.int32(17): 0, np.int32(18): 2, np.int32(19): 2, np.int32(20): 2, np.int32(21): 2, np.int32(22): 2, np.int32(23): 2, np.int32(24): 3, np.int32(25): 3, np.int32(26): 3, np.int32(27): 3, np.int32(28): 3, np.int32(29): 3, np.int32(30): 3, np.int32(31): 3, np.int32(32): 3, np.int32(33): 3, np.int32(34): 3, np.int32(35): 3, np.int32(36): 1, np.int32(37): 1, np.int32(38): 1, np.int32(39): 1, np.int32(40): 1, np.int32(41): 1, np.int32(42): 2, np.int32(43): 2, np.int32(44): 2, np.int32(45): 2, np.int32(46): 2, np.int32(47): 2, np.int32(48): 2, np.int32(49): 2, np.int32(50): 2, np.int32(51): 2, np.int32(52): 2, np.int32(53): 2, np.int32(54): 3, np.int32(55): 3, np.int32(56): 3, np.int32(57): 3, np.int32(58): 3, np.int

In [13]:
# Create the list of channels to correct
channels_to_correct = [ch for ch, value in channel_dict.items() if value in {0, 1}]
print(channels_to_correct)

# Create a mask for the channels to correct
mask = ak.Array([
    [[ch in channels_to_correct for ch in ophit] for ophit in flash] for flash in f_ophit_ch_2
])

# Apply the mask to f_ophit_t variable
f_ophit_t_adj = ak.where(mask, f_ophit_t_2 - 0.135, f_ophit_t_2)

[np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(36), np.int32(37), np.int32(38), np.int32(39), np.int32(40), np.int32(41), np.int32(60), np.int32(61), np.int32(62), np.int32(63), np.int32(64), np.int32(65), np.int32(66), np.int32(67), np.int32(68), np.int32(69), np.int32(70), np.int32(71), np.int32(84), np.int32(85), np.int32(86), np.int32(87), np.int32(88), np.int32(89), np.int32(90), np.int32(91), np.int32(92), np.int32(93), np.int32(94), np.int32(95), np.int32(114), np.int32(115), np.int32(116), np.int32(117), np.int32(118), np.int32(119), np.int32(138), np.int32(139), np.int32(140), np.int32(141), np.int32(142), np.int32(143), np.int32(144), np.int32(145), np.int32(146), np.int32(147), np.int32(148), np.int32(149), np.int32(162), np.int32(163), np.int32(164), np.int32(165), np.int32(166), np.int32(167), np.int32(168), np.int32(169), np.int32(170), np.int32(1

In [14]:
del f_ophit_t_2

### Selección de TPC para la variable dEprom{x,y,z} y (Opcional) para los flashes de las variables flash_ophit

In [15]:
import awkward as ak
import numpy as np

# --- 1. Clasificación de canales ---
pmt_channels = [ch for ch, val in channel_dict.items() if val in {0, 1}]
xas_channels = [ch for ch, val in channel_dict.items() if val in {2, 3}]

def split_even_odd(channels):
    return set(filter(lambda x: x % 2 == 0, channels)), set(filter(lambda x: x % 2 != 0, channels))

pmt_even, pmt_odd = split_even_odd(pmt_channels)
xas_even, xas_odd = split_even_odd(xas_channels)

def categorize_first_ch(vector):
    if isinstance(vector, (list, ak.Array)) and len(vector) > 0:
        ch = vector[0]
        if ch in pmt_even: return 0
        if ch in pmt_odd:  return 1
        if ch in xas_even: return 2
        if ch in xas_odd:  return 3
    return -1  # No clasificado o vacío

categorized_flashes = ak.Array([
    [categorize_first_ch(flash) for flash in event]
    for event in ak.to_list(f_ophit_ch_2)
])

# --- 2. Sumar los PEs de cada flash ---
sum_pe = ak.sum(f_ophit_PE_2, axis=2)  # [evento][flash]

# --- 3. Crear máscaras por categoría ---
mask_even = (categorized_flashes == 0) | (categorized_flashes == 2)
mask_odd  = (categorized_flashes == 1) | (categorized_flashes == 3)

# --- 4. Sumar PE por grupo ---
sum_even = ak.sum(ak.where(mask_even, sum_pe, 0), axis=1)
sum_odd  = ak.sum(ak.where(mask_odd, sum_pe, 0), axis=1)

# --- 5. Selección del grupo con mayor PE si hay más de 2 flashes ---
n_flashes = ak.num(categorized_flashes)
decision = sum_even >= sum_odd 

# Generar máscara de selección
selected_mask = ak.Array([
    np.ones(n, dtype=bool) if n <= 2  # Keep all flashes for ≤ 2
    else (mask_even[i] if decision[i] else mask_odd[i])
    for i, n in enumerate(ak.to_list(n_flashes))
])

# --- 6. Función para filtrar un array usando la máscara ---
def filter_by_mask(array, mask):
    return ak.Array([
        [item for item, flag in zip(event, event_mask) if flag]
        for event, event_mask in zip(ak.to_list(array), ak.to_list(mask))
    ])

# --- 7. Aplicar máscaras de selección ---
f_ophit_t_adj_sel       = filter_by_mask(f_ophit_t_adj, selected_mask)
f_ophit_PE_2_sel        = filter_by_mask(f_ophit_PE_2, selected_mask)
f_ophit_ch_2_sel        = filter_by_mask(f_ophit_ch_2, selected_mask)
categorized_flashes_sel = filter_by_mask(categorized_flashes, selected_mask)

# --- 8. Selección TPC ganadora ---
selector = ak.Array([[d, not d] for d in decision])

def select_tpc_value(array_2d, selector):
    return ak.sum(ak.where(selector, array_2d, 0), axis=1)

dEpromx_sel = select_tpc_value(dEpromx_2, selector)
dEpromy_sel = select_tpc_value(dEpromy_2, selector)
dEpromz_sel = select_tpc_value(dEpromz_2, selector)
dEtpc_sel   = select_tpc_value(dEtpc_2, selector)

In [16]:
del f_ophit_t_adj, f_ophit_PE_2, f_ophit_ch_2, categorized_flashes, dEpromx_2, dEpromy_2, dEpromz_2, dEtpc_2, decision, mask_even, mask_odd, sum_even, sum_odd, n_flashes

### Para reconstrucción temporal

In [24]:
# Create a boolean mask where dEpromx_f_unique is not -999 & select events with deposition >200 MeV (dEtpc_f > 200)

mask = (dEpromx_sel != -999) & (dEpromy_sel != -999) & (dEpromz_sel != -999) & (dEtpc_sel > 200)
mask_1d = ak.to_numpy(mask)

# Apply the mask to both the image and dEpromx_f_unique to keep only the valid entries

nuvT_3 = nuvT_2[mask_1d]
f_ophit_PE_3 = f_ophit_PE_2_sel[mask_1d]
f_ophit_ch_3 = f_ophit_ch_2_sel[mask_1d]
f_ophit_t_3 = f_ophit_t_adj_sel[mask_1d]
dEpromx_3 = dEpromx_sel[mask_1d]
dEpromy_3 = dEpromy_sel[mask_1d]
dEpromz_3 = dEpromz_sel[mask_1d]
dEtpc_3 = dEtpc_sel[mask_1d]
nuvZ_3 = nuvZ_2[mask_1d]

### Para reconstrucción espacial

In [26]:
# Create a boolean mask where dEpromx_f_unique is not -999

mask = (dEpromx_sel != -999) & (dEpromy_sel != -999) & (dEpromz_sel != -999)
mask_1d = ak.to_numpy(mask)

# Apply the mask to both the image and dEpromx_f_unique to keep only the valid entries

nuvT_3 = nuvT_2[mask_1d]
f_ophit_PE_3 = f_ophit_PE_2_sel[mask_1d]
f_ophit_ch_3 = f_ophit_ch_2_sel[mask_1d]
f_ophit_t_3 = f_ophit_t_adj_sel[mask_1d]
dEpromx_3 = dEpromx_sel[mask_1d]
dEpromy_3 = dEpromy_sel[mask_1d]
dEpromz_3 = dEpromz_sel[mask_1d]
dEtpc_3 = dEtpc_sel[mask_1d]
nuvZ_3 = nuvZ_2[mask_1d]

In [19]:
print(len(nuvT_2))

101189


In [27]:
print(len(nuvT_3))

100791


In [28]:
del nuvT_2, f_ophit_PE_2_sel, f_ophit_ch_2_sel, f_ophit_t_adj_sel, nuvZ_2

In [29]:
# Diccionario con los arrays
data = {
    "nuvT": nuvT_3,
    "f_ophit_PE": f_ophit_PE_3,
    "f_ophit_ch": f_ophit_ch_3,
    "f_ophit_t": f_ophit_t_3,
    "dEpromx": dEpromx_3,
    "dEpromy": dEpromy_3,
    "dEpromz": dEpromz_3,
    "nuvZ": nuvZ_3,
}

# Convertir a Arrow Table y guardar como Parquet
table = ak.to_arrow_table(data)
pq.write_table(table, "/exp/sbnd/app/users/svidales/AI_nuvT/v0504_tpcselection_preproc_springval_allevents_nonenergylimit.parquet")

In [37]:
print(len(f_ophit_PE_3))

75983
